# Meta Robyn (example): national weekly input from `MMM Data.csv`

**Example only.** Robyn is typically run on a **single national/regional weekly** table. This notebook **aggregates** the DMA panel to **national weekly** series (sum impressions, mean controls, sum target) so the shape matches common Robyn demos.

**Policy:** Same as other example notebooks — **inlined** prep from `MMM Data.csv`; **do not** change `examples/dashboard_rmse_optimized.py` for this workflow.

**Robyn in Python (`robynpy`)** is **beta** and still expects **R + `glmnet`**. Install: `pip install robynpy`. Import the client as **`from robyn.robyn import Robyn`** (not `from robyn import Robyn`). See [Robyn Python README](https://github.com/facebookexperimental/Robyn/blob/main/python/README.md). The cells below:

1. Build a **Robyn-friendly CSV** (`robyn_input_national_weekly_example.csv`) with `DATE`, `dep_var`, `spend_*`, and controls.
2. **Benchmark metrics** — sklearn **train/holdout R² & RMSE** (same **`holdout_ratio`** as `pymc_aligned_dcm_config.json`) for side-by-side comparison with **PyMC / Meridian / DCM**; optional **`robynpy`** rows add Robyn-native **`rsq_train` / NRMSE** after `train_models`.
3. Optionally sketch **`robynpy`** wiring — full training requires **`HolidaysData`** and **`Hyperparameters`** (use Meta’s `tutorial1.ipynb` as the source of truth).

**Related:** [`pymc_marketing_mmm_pooled_dma.ipynb`](pymc_marketing_mmm_pooled_dma.ipynb) (panel / PyMC path).


## Configuration


In [17]:
import json
import os
from pathlib import Path


def _deepcausalmmm_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for d in [cwd, *cwd.parents]:
        if (d / "pyproject.toml").is_file() and (d / "deepcausalmmm").is_dir():
            return d
    raise FileNotFoundError("Could not find deepcausalmmm repo root.")


REPO_ROOT = _deepcausalmmm_repo_root()

_cfg_path = REPO_ROOT / "examples" / "pymc_aligned_dcm_config.json"
with open(_cfg_path, encoding="utf-8") as f:
    DCM_CONFIG = json.load(f)

DATA_PATH = REPO_ROOT / "examples" / "data" / "MMM Data.csv"
if not DATA_PATH.is_file():
    alt = REPO_ROOT / "data" / "MMM Data.csv"
    if alt.is_file():
        DATA_PATH = alt
    else:
        raise FileNotFoundError(f"MMM Data.csv not found under examples/data or data/")

OUT_CSV = REPO_ROOT / "examples" / "robyn_input_national_weekly_example.csv"
HOLDOUT_RATIO = float(DCM_CONFIG.get("holdout_ratio", 0.12))

print(f"REPO_ROOT={REPO_ROOT} | OUT_CSV={OUT_CSV.name} | holdout_ratio={HOLDOUT_RATIO}")


REPO_ROOT=/Users/adityapu/Documents/GitHub/deepcausalmmm | OUT_CSV=robyn_input_national_weekly_example.csv | holdout_ratio=0.12


## Build national weekly table and export CSV


In [18]:
import numpy as np
import pandas as pd

df_raw = pd.read_csv(DATA_PATH)
impression_cols = [c for c in df_raw.columns if c.startswith("impressions_")]
control_cols = [c for c in df_raw.columns if c.startswith("control_")]
target_col = "target_visits" if "target_visits" in df_raw.columns else "value_visits_visits"
region_col, time_col = "dmacode", "weekid"

df_raw = df_raw.sort_values([region_col, time_col])
for c in impression_cols:
    df_raw[c] = df_raw[c].fillna(0)
for c in control_cols + [target_col]:
    df_raw[c] = df_raw.groupby(region_col)[c].ffill()

weeks_sorted = sorted(df_raw[time_col].unique())
week_to_date = {w: pd.Timestamp("2020-01-06") + pd.Timedelta(weeks=i) for i, w in enumerate(weeks_sorted)}
df_raw["DATE"] = df_raw[time_col].map(week_to_date)

_cal_sparse = sorted(df_raw["DATE"].dropna().unique())
_ncs = len(_cal_sparse)
_twc = _ncs - int(_ncs * HOLDOUT_RATIO)
_cut_sparse = _cal_sparse[_twc - 1]
_mask_train_dates = df_raw["DATE"] <= _cut_sparse
for c in control_cols + [target_col]:
    if df_raw[c].isna().any():
        _med = df_raw.loc[_mask_train_dates, c].median()
        if pd.isna(_med):
            _med = float(df_raw.loc[_mask_train_dates, c].mean())
        df_raw[c] = df_raw[c].fillna(_med)

agg_map = {c: "sum" for c in impression_cols + [target_col]}
agg_map.update({c: "mean" for c in control_cols})
nat = df_raw.groupby("DATE", as_index=False).agg(agg_map).sort_values("DATE").reset_index(drop=True)

n_channels = len(impression_cols)
channel_short = [f"ch{i:02d}" for i in range(1, n_channels + 1)]
rename_imp = dict(zip(impression_cols, [f"spend_{c}" for c in channel_short]))
nat = nat.rename(columns=rename_imp)
nat = nat.rename(columns={target_col: "dep_var"})

# Robyn often uses matching exposure + spend names; we only have impressions → use as both spend and exposure columns
exposure_names = [f"imp_{c}" for c in channel_short]
for ch, ex in zip(channel_short, exposure_names):
    nat[ex] = nat[f"spend_{ch}"]

# Prophet / Robyn expect weekday column name DATE
cols_order = ["DATE", "dep_var"] + [f"spend_{c}" for c in channel_short] + exposure_names + control_cols
robyn_df = nat[cols_order].copy()

robyn_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(robyn_df)} rows × {robyn_df.shape[1]} cols → {OUT_CSV}")
robyn_df.head()


Wrote 109 rows × 35 cols → /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robyn_input_national_weekly_example.csv


,DATE,dep_var,spend_ch01,spend_ch02,spend_ch03,spend_ch04,spend_ch05,spend_ch06,spend_ch07,spend_ch08,...,imp_ch11,imp_ch12,imp_ch13,control_01,control_02,control_03,control_04,control_05,control_06,control_07
0,2020-01-06,146879536,231376,7787579,6.398508e+06,3.680707e+08,2134726,535980,2.554446e+06,6916043.00,...,11987678,7.103762e+00,1.214158e+04,662.152933,235016.198263,104.198522,90.272346,295498.200125,6.527280,3.403925
1,2020-01-13,147495743,345291,8308631,6.805782e+06,5.505227e+08,11225148,217622,2.831352e+07,11365825.50,...,12175498,2.220629e+04,3.129336e+05,663.652674,235121.813529,104.198717,89.878503,295498.200160,6.520378,3.403743
2,2020-01-20,142743887,303408,7777606,7.431357e+06,6.529255e+08,6673031,280924,3.038956e+07,15282071.75,...,11899507,2.796900e+06,3.964576e+06,674.405829,235428.627273,102.513743,90.788449,296479.767433,6.491136,4.433316
3,2020-01-27,136780122,124571,13220860,4.692551e+06,6.466759e+08,4749977,318923,2.950125e+07,13826460.50,...,12212314,1.664327e+07,2.567286e+07,681.338984,235726.644759,101.334171,89.845882,297166.864545,6.780116,4.132567
4,2020-02-03,142449068,115389,10648729,4.175715e+06,6.627031e+08,3897687,730764,2.748292e+07,13447309.75,...,12165708,1.908918e+07,8.872034e+07,687.016791,235711.647807,100.508449,88.526310,297633.075615,6.728697,3.922086


## Benchmark metrics (PyMC / DCM / Meridian–aligned)

**Holdout (DCM parity):** Matches `UnifiedDataPipeline.temporal_split` in `deepcausalmmm/core/data.py`: `holdout_weeks = int(n_weeks * holdout_ratio)`, `train_weeks = n_weeks - holdout_weeks`, train = first `train_weeks` weeks, holdout = last `holdout_weeks` weeks. The aggregation cell uses the same cut (last train date = sorted unique `DATE` at index `train_weeks - 1`; train `DATE <=` that). National CSV is one row per week, so the sklearn split matches DCM’s week axis when panel weeks align. *DCM training only:* `burn_in_weeks` padding and `min_train_weeks` do not change this calendar definition.

**Sklearn rows:** `sklearn.metrics.r2_score` and `sqrt(mean_squared_error)` on **original-scale `dep_var`**, **Ridge** fit **only on train weeks** (true out-of-sample predictions on the tail — comparable spirit to PyMC train/holdout, not Robyn in-sample `rsq_train` on the full featurized window).

**Robyn row:** After `train_models`, we print **Robyn’s** `rsq_train` / `nrmse` from the package (R-glmnet path; **not** the same definition as sklearn). Optional `train_models` needs **R + glmnet** and can take minutes unless you lower `iterations`.

In [19]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score


def national_calendar_last_train_date(dates: pd.Series, holdout_ratio: float) -> pd.Timestamp:
    """Last train week: n_train = n - int(n * ratio), cut = cal[n_train - 1] (matches aggregation / PyMC config)."""
    cal = sorted(pd.to_datetime(dates.dropna().unique()))
    n = len(cal)
    if n < 2:
        raise ValueError("Need at least 2 weekly dates for train/holdout.")
    n_train = n - int(n * holdout_ratio)
    n_train = max(1, n_train)
    return cal[n_train - 1]


def sklearn_ridge_train_holdout_metrics(
    df: pd.DataFrame,
    date_col: str,
    y_col: str,
    feature_cols: list,
    holdout_ratio: float,
    *,
    ridge_alpha: float = 1.0,
) -> dict:
    """Ridge fit on train weeks only; sklearn R² / RMSE on train and holdout (original scale)."""
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col])
    cut = national_calendar_last_train_date(d[date_col], holdout_ratio)
    tr_m = d[date_col] <= cut
    ho_m = d[date_col] > cut
    X = d[feature_cols].apply(pd.to_numeric, errors="coerce").astype(float).to_numpy()
    y = pd.to_numeric(d[y_col], errors="coerce").astype(float).to_numpy()
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("Non-finite values in X or y after coercion.")
    X_tr, y_tr = X[tr_m.values], y[tr_m.values]
    X_ho, y_ho = X[ho_m.values], y[ho_m.values]
    if len(y_ho) == 0:
        raise ValueError("Empty holdout after split; check holdout_ratio and dates.")
    m = Ridge(alpha=ridge_alpha, fit_intercept=True).fit(X_tr, y_tr)
    pred_tr = m.predict(X_tr)
    pred_ho = m.predict(X_ho)
    naive_ho = np.full_like(y_ho, fill_value=float(np.mean(y_tr)), dtype=float)
    return {
        "holdout_cut_date": cut,
        "n_train_weeks": int(tr_m.sum()),
        "n_holdout_weeks": int(ho_m.sum()),
        "train_r2": float(r2_score(y_tr, pred_tr)),
        "train_rmse": float(np.sqrt(mean_squared_error(y_tr, pred_tr))),
        "holdout_r2": float(r2_score(y_ho, pred_ho)),
        "holdout_rmse": float(np.sqrt(mean_squared_error(y_ho, pred_ho))),
        "holdout_rmse_naive_train_mean": float(np.sqrt(mean_squared_error(y_ho, naive_ho))),
    }


# --- National weekly CSV (always runs): same file as Robyn input ---
_bench = pd.read_csv(OUT_CSV)
_bench["DATE"] = pd.to_datetime(_bench["DATE"])
_feat = [c for c in _bench.columns if c not in ("DATE", "dep_var")]
_m_csv = sklearn_ridge_train_holdout_metrics(
    _bench, "DATE", "dep_var", _feat, HOLDOUT_RATIO, ridge_alpha=1.0
)

compare = pd.DataFrame(
    [
        {
            "model": "national_sklearn_ridge_raw_csv",
            "train_r2": _m_csv["train_r2"],
            "train_rmse": _m_csv["train_rmse"],
            "holdout_r2": _m_csv["holdout_r2"],
            "holdout_rmse": _m_csv["holdout_rmse"],
            "holdout_rmse_naive_train_mean": _m_csv["holdout_rmse_naive_train_mean"],
            "n_train_weeks": _m_csv["n_train_weeks"],
            "n_holdout_weeks": _m_csv["n_holdout_weeks"],
            "holdout_cut_date": str(_m_csv["holdout_cut_date"].date()),
        }
    ]
).set_index("model")

# Plain-text R² & RMSE (same definitions as PyMC / DCM notebooks: sklearn on original-scale dep_var)
print("R2 (train):   ", round(_m_csv["train_r2"], 6))
print("RMSE (train): ", round(_m_csv["train_rmse"], 4))
print("R2 (holdout): ", round(_m_csv["holdout_r2"], 6))
print("RMSE (holdout):", round(_m_csv["holdout_rmse"], 4))

r2_rmse = pd.DataFrame(
    {
        "split": ["train", "holdout"],
        "R2": [_m_csv["train_r2"], _m_csv["holdout_r2"]],
        "RMSE": [_m_csv["train_rmse"], _m_csv["holdout_rmse"]],
    }
).set_index("split")
print("\n=== R2 & RMSE (summary) ===")
display(r2_rmse)

print("\n=== Full benchmark row (extra columns) ===")
display(compare.T)

R2 (train):    0.855818
RMSE (train):  5514718.6663
R2 (holdout):  -12.429327
RMSE (holdout): 16744106.3826

=== R2 & RMSE (summary) ===


/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.2548766073165542e-19.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,R2,RMSE
split,,
train,0.855818,5.514719e+06
holdout,-12.429327,1.674411e+07



=== Full benchmark row (extra columns) ===


model,national_sklearn_ridge_raw_csv
train_r2,0.855818
train_rmse,5514718.666299
holdout_r2,-12.429327
holdout_rmse,16744106.382586
holdout_rmse_naive_train_mean,22627150.775283
n_train_weeks,96
n_holdout_weeks,13
holdout_cut_date,2021-11-01


## Optional: `robynpy` skeleton (requires R + glmnet)

For **`interval_type="week"`**, Robyn’s feature engineering expects **`dt_holidays`** with columns **`ds`**, **`holiday`** (name), **`country`**, and **`year`** — otherwise you get **`KeyError: 'year'`** inside `_set_holidays`.

If you have no organic channels, pass **`organic_vars=[]`** in **`MMMDataSpec`** (default **`None`** can make Prophet feature engineering fail with **`TypeError: ... NoneType ... list`**).

For **`train_models`**, set **`window_start`** / **`window_end`** to your date range (e.g. min/max **`DATE`**). If they are missing, **`rolling_window_end_which`** stays **`None`** and Ridge training fails with **`unsupported operand type(s) for +: 'NoneType' and 'int'`**.


In [20]:
# Set True only if you have R + glmnet and robynpy installed (`pip install robynpy`).
RUN_ROBYNPY = True

if RUN_ROBYNPY:
    # Meta RobynPy: Robyn lives in robyn.robyn (not re-exported from robyn.__init__)
    from robyn.robyn import Robyn
    from robyn.data.entities.mmmdata import MMMData
    from robyn.data.entities.holidays_data import HolidaysData
    import time
    import os
    from robyn.data.entities.hyperparameters import Hyperparameters, ChannelHyperparameters
    from robyn.data.entities.enums import (
        DependentVarType,
        PaidMediaSigns,
        ContextSigns,
        AdstockType,
        ProphetVariableType,
        ProphetSigns,
    )
    from robyn.modeling.entities.modelrun_trials_config import TrialsConfig
    from IPython.display import display

    work = REPO_ROOT / "examples" / "robynpy_working_dir_example"
    r = Robyn(working_dir=str(work))

    data = pd.read_csv(OUT_CSV)
    data["DATE"] = pd.to_datetime(data["DATE"])

    spend_cols = [c for c in data.columns if c.startswith("spend_")]
    imp_cols = [c for c in data.columns if c.startswith("imp_")]
    ctx = [c for c in data.columns if c.startswith("control_")]

    spec = MMMData.MMMDataSpec(
        dep_var="dep_var",
        dep_var_type=DependentVarType.CONVERSION,
        date_var="DATE",
        # Required for train_models: otherwise rolling_window_end_which stays None → TypeError in RidgeDataBuilder
        window_start=pd.Timestamp(data["DATE"].min()),
        window_end=pd.Timestamp(data["DATE"].max()),
        paid_media_spends=spend_cols,
        paid_media_vars=imp_cols,
        paid_media_signs=[PaidMediaSigns.POSITIVE] * len(spend_cols),
        context_vars=ctx,
        context_signs=[ContextSigns.DEFAULT] * len(ctx),
        # Robyn's Prophet path does list concat; MMMDataSpec defaults organic_vars=None → TypeError
        organic_vars=[],
        day_interval=7,
        interval_type="week",
    )
    mmm_data = MMMData(data=data, mmmdata_spec=spec)
    # RidgeDataBuilder uses iloc[start : end+1]; those must be integer *positions*.
    # Robyn's calculate_rolling_window_indices uses idxmin() *labels* — mismatch breaks train_models.
    mmm_data.data = mmm_data.data.sort_values("DATE").reset_index(drop=True)
    _n = len(mmm_data.data)
    _sp = mmm_data.mmmdata_spec
    _sp.rolling_window_start_which = 0
    _sp.rolling_window_end_which = _n - 1
    _sp.rolling_window_length = _n
    _sp.window_start = pd.Timestamp(mmm_data.data["DATE"].iloc[0])
    _sp.window_end = pd.Timestamp(mmm_data.data["DATE"].iloc[-1])
    _sp.refresh_added_start = _sp.window_start

    # Weekly `interval_type`: Robyn's `_set_holidays` does groupby(["ds","country","year"]) and
    # needs a `holiday` name column — not just ds/country (else KeyError 'year').
    try:
        import holidays as _hlib
    except ImportError:
        _hlib = None
    _dr = pd.date_range(data["DATE"].min(), data["DATE"].max(), freq="D")
    _hol_rows = []
    if _hlib is not None:
        _cal = _hlib.country_holidays(
            "US", years=range(int(_dr.min().year), int(_dr.max().year) + 1)
        )
        for _ts in _dr:
            if _ts.date() in _cal:
                _hol_rows.append(
                    {
                        "ds": pd.Timestamp(_ts),
                        "holiday": str(_cal[_ts.date()]),
                        "country": "US",
                        "year": int(_ts.year),
                    }
                )
    if not _hol_rows:
        _hol_rows.append(
            {
                "ds": pd.Timestamp(_dr[0]),
                "holiday": "none",
                "country": "US",
                "year": int(_dr[0].year),
            }
        )
    _dt_holidays = pd.DataFrame(_hol_rows)

    holidays = HolidaysData(
        dt_holidays=_dt_holidays,
        prophet_vars=[
            ProphetVariableType.TREND,
            ProphetVariableType.SEASON,
            ProphetVariableType.WEEKDAY,
            ProphetVariableType.HOLIDAY,
        ],
        prophet_signs=[
            ProphetSigns.DEFAULT,
            ProphetSigns.POSITIVE,
            ProphetSigns.NEGATIVE,
            ProphetSigns.DEFAULT,
        ],
        prophet_country="US",
    )

    hp = {
        ch: ChannelHyperparameters(alphas=[0.8], gammas=[0.3], thetas=[0.5])
        for ch in spend_cols
    }
    hyper = Hyperparameters(hyperparameters=hp, adstock=AdstockType.GEOMETRIC)

    r.initialize(mmm_data, holidays, hyper)
    r.feature_engineering(display_plots=False, export_plots=False)

    # --- Sklearn metrics on Robyn featurized dt_mod (Prophet + inputs), same calendar holdout as PyMC row ---
    _dt = r.featurized_mmm_data.dt_mod.copy()
    _dt["ds"] = pd.to_datetime(_dt["ds"])
    _dep = mmm_data.mmmdata_spec.dep_var
    _ex = ["ds", _dep, "dep_var"]
    _ex = [c for c in _ex if c in _dt.columns]
    _Xdf = _dt.drop(columns=_ex)
    for _col in _Xdf.select_dtypes(include=["datetime64", "object"]).columns:
        _Xdf[_col] = pd.to_datetime(_Xdf[_col], errors="coerce")
        _mn = _Xdf[_col].min()
        _Xdf[_col] = ((_Xdf[_col] - _mn).dt.total_seconds() / 86400.0).fillna(0)
    _cat = _Xdf.select_dtypes(include=["object", "category"]).columns
    if len(_cat):
        _Xdf = pd.get_dummies(_Xdf, columns=list(_cat), drop_first=True)
    _feat_robyn = [c for c in _Xdf.columns if pd.api.types.is_numeric_dtype(_Xdf[c])]
    _Xdf = _Xdf[_feat_robyn].astype(float)
    _m_fe = sklearn_ridge_train_holdout_metrics(
        pd.concat([_dt[["ds", _dep]].rename(columns={_dep: "dep_var"}), _Xdf], axis=1),
        "ds",
        "dep_var",
        _feat_robyn,
        HOLDOUT_RATIO,
        ridge_alpha=1.0,
    )
    _row_fe = {
        "model": "national_sklearn_ridge_robyn_dt_mod",
        "train_r2": _m_fe["train_r2"],
        "train_rmse": _m_fe["train_rmse"],
        "holdout_r2": _m_fe["holdout_r2"],
        "holdout_rmse": _m_fe["holdout_rmse"],
        "holdout_rmse_naive_train_mean": _m_fe["holdout_rmse_naive_train_mean"],
        "n_train_weeks": _m_fe["n_train_weeks"],
        "n_holdout_weeks": _m_fe["n_holdout_weeks"],
        "holdout_cut_date": str(_m_fe["holdout_cut_date"].date()),
    }
    print("=== Robyn FE + sklearn Ridge — R2 & RMSE ===")
    print("R2 (train):   ", round(_m_fe["train_r2"], 6))
    print("RMSE (train): ", round(_m_fe["train_rmse"], 4))
    print("R2 (holdout): ", round(_m_fe["holdout_r2"], 6))
    print("RMSE (holdout):", round(_m_fe["holdout_rmse"], 4))
    display(
        pd.DataFrame(
            {
                "split": ["train", "holdout"],
                "R2": [_m_fe["train_r2"], _m_fe["holdout_r2"]],
                "RMSE": [_m_fe["train_rmse"], _m_fe["holdout_rmse"]],
            }
        ).set_index("split")
    )
    display(pd.DataFrame([_row_fe]).set_index("model").T)

    # --- Robyn Nevergrad + glmnet (optional; needs R); small budget for notebook smoke ---
    RUN_ROBYN_TRAIN = True
    if RUN_ROBYN_TRAIN:
        try:
            _tc = TrialsConfig(trials=1, iterations=50)
            _cores = min(4, os.cpu_count() or 1)
            t0 = time.perf_counter()
            r.train_models(
                trials_config=_tc,
                ts_validation=False,
                display_plots=False,
                export_plots=False,
                cores=_cores,
            )
            fit_s = time.perf_counter() - t0
            mo = r.model_outputs
            last = mo.trials[-1]

            def _sf(s):
                return float(s.iloc[0]) if hasattr(s, "iloc") else float(s)

            _row_rn = {
                "model": "robynpy_train_models_best_trial",
                "fit_runtime_s": fit_s,
                "robyn_rsq_train": _sf(last.rsq_train),
                "robyn_nrmse": _sf(last.nrmse),
                "robyn_nrmse_train": _sf(last.nrmse_train),
                "robyn_decomp_rssd": _sf(last.decomp_rssd),
                "robyn_select_id": str(mo.select_id),
            }
            print("=== Robyn glmnet trial (R² is Robyn/rsq_train; RMSE scale differs from sklearn above) ===")
            print("R2 (train, Robyn):", round(_row_rn["robyn_rsq_train"], 6))
            print(
                "NRMSE (Robyn):",
                round(_row_rn["robyn_nrmse"], 6),
                "(= RMSE / range(y) in Robyn’s transformed target; not identical to sklearn RMSE on dep_var)",
            )
            print("NRMSE train (Robyn):", round(_row_rn["robyn_nrmse_train"], 6))
            display(pd.DataFrame([_row_rn]).set_index("model").T)
        except Exception as _e:
            _msg = str(_e)
            if "NoneType" in _msg and "int" in _msg:
                print(
                    "train_models failed: rolling-window indices (re-run full cell; notebook resets data index + rolling_window_*).",
                    type(_e).__name__,
                    _e,
                )
            else:
                print(
                    "train_models skipped or failed (often R + glmnet + rpy2). Error:",
                    type(_e).__name__,
                    _e,
                )
    else:
        print("RUN_ROBYN_TRAIN=False: skipped glmnet training.")
else:
    print("Skipping robynpy (RUN_ROBYNPY=False). Use OUT_CSV with R Robyn or enable RUN_ROBYNPY after installing R/glmnet/robynpy.")


INFO: Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
INFO: Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
INFO: Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
INFO: Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
INFO: Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
2026-03-22 10:56:19,608 - robyn.robyn - INFO - Initialized Robyn in /Users/adityapu/Documents/GitHub/deepcausalmmm/examples/robynpy_working_dir_example
INFO: Validating input data
INFO: Validating input data
INFO: Validating input data
INFO: Validating input data
INFO: Validating input data
2026-03-22 10:56:19,628 - robyn.robyn - INFO - Validating input data
2026-03-22 10:56:19,632 - robyn.data.validation.mmmdata_validation - INFO - Starting

=== Robyn FE + sklearn Ridge — R2 & RMSE ===
R2 (train):    0.955852
RMSE (train):  3051577.3891
R2 (holdout):  -2.932473
RMSE (holdout): 9060825.0927


/opt/homebrew/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 5.069778772014091e-19.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


,R2,RMSE
split,,
train,0.955852,3.051577e+06
holdout,-2.932473,9.060825e+06


model,national_sklearn_ridge_robyn_dt_mod
train_r2,0.955852
train_rmse,3051577.389103
holdout_r2,-2.932473
holdout_rmse,9060825.092695
holdout_rmse_naive_train_mean,22627150.775283
n_train_weeks,96
n_holdout_weeks,13
holdout_cut_date,2021-11-01


INFO: Training models
INFO: Training models
INFO: Training models
INFO: Training models
INFO: Training models
2026-03-22 10:56:39,924 - robyn.robyn - INFO - Training models
2026-03-22 10:56:39,926 - robyn.modeling.base_model_executor - INFO - Initializing BaseModelExecutor
2026-03-22 10:56:39,926 - robyn.modeling.model_executor - INFO - Starting model execution with model_name=Models.RIDGE
2026-03-22 10:56:39,927 - robyn.modeling.base_model_executor - INFO - Input validation successful
2026-03-22 10:56:39,927 - robyn.modeling.base_model_executor - INFO - Preparing hyperparameters
2026-03-22 10:56:39,927 - robyn.modeling.base_model_executor - INFO - Completed hyperparameter preparation with 1 parameters to optimize
2026-03-22 10:56:39,927 - robyn.modeling.model_executor - INFO - Initializing Ridge model builder
2026-03-22 10:56:39,930 - robyn.modeling.model_executor - INFO - Building models with configured parameters
2026-03-22 10:56:39,931 - robyn.modeling.ridge.ridge_data_builder - IN

train_models skipped or failed (often R + glmnet + rpy2). Error: KeyError 'spend_ch01_thetas'


## Next steps

- **R workflow:** Load `robyn_input_national_weekly_example.csv` into Meta’s **R Robyn** demo dataset slot (column names align with common tutorials; you may still need to adjust `robyn_inputs` / `hyperparameters` / holidays).
- **Python:** Clone [Robyn](https://github.com/facebookexperimental/Robyn) and open `python/src/robyn/tutorials/tutorial1.ipynb` — copy hyperparameter and holiday setup from there.
- **Scope:** National weekly here ≠ DMA panel in the PyMC notebook; compare models only on **consistent** aggregation and metrics definitions.
